In [1]:
# 04 Run CHMv2 Minimal
# Goal: run CHMv2 on the native-resolution RGB input with minimal preprocessing.


In [3]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

import rasterio
from rasterio.windows import Window

import torch
import torch.nn.functional as F
from torchvision.transforms import v2
from tqdm.notebook import tqdm


In [5]:
from pathlib import Path
from urllib.request import urlretrieve

project_root = Path("/Users/rgcappl037/Documents/New project/electric-pole-analysis")
weights_dir = project_root / "data" / "weights" / "chmv2"
weights_dir.mkdir(parents=True, exist_ok=True)

backbone_weights = weights_dir / "dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
chmv2_weights = weights_dir / "dinov3_vitl16_chmv2_dpt_head-3703d643.pth"

backbone_url = "https://dinov3.llamameta.net/dinov3_vitl16/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiMDh4bnAyM2l2YWFnajVjNmlnZzlrMTlkIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NzQ4MjA4MzV9fX1dfQ__&Signature=LJw5drwhv9GJneFEAv1%7EjTId1a3cNYBNaq5K%7E5cnSrsFM95BzxkOdElIJ2ZiNe1PJHtNe8hPjtNMH4Nn28kwjkXwaEpvM0pOnB77N6eEbnuQsYRlpbwrredgCYhO33tlz9kSBCRzj2ZIEpiGgueYdTyA79hP8ZzvMVB9Px4QWGLE2sX0n14opjb2SW7l8FLjTJkZbadQTRDMPbmX3WjfxQTjhAoEh9LDgenQPxi-cDYVq1fYqYDfrOWQ-p%7EIKIufeOn9RJtO9BsbC9dDBhFcyfEIorAQ2hv6e-qsxL1MwveLJiM2QVnhN8mgc%7Ef2LDIQAFmzHEctkeyRdpxJHvsy9g__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1574019413687540"
chmv2_url = "https://dinov3.llamameta.net/chmv2/dinov3_vitl16_chmv2_dpt_head-3703d643.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiMHUzdXZvMWUzbXZ4aGRlaGN4c2tqempvIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NzQ4MjA5MTB9fX1dfQ__&Signature=b4165rS8z4XR8czmtjK1O5VgfkTT%7Egc9FtcKUFcFt1aHvkn21qdixWaPU%7EIlhGvGQa3Hf3awL0wEyFpoGhTmQJCNmBrQf55L3jBPU%7EWPI9CP-BfMaQSiD5nGS-rodbTDRmhh7lx%7ELrdvrtmrQZGoRW5AjAel3rvGuTPq77HRE9to5cc7q4ExxqGQFVwZHAp06jmhxXCpWNNFVSORL7OlUTeVADfIscR9mJrdxSQLuPSeZPx38znUZKeh9tTLn1oIa1Kcrj2ujsOOQe1qStwkkEdqwvESjq3A8XoJp-Tprlim36y1yauxnxVKqMmrJbXCSpUDIPFucCKzhhy-i2pRLw__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1257385482634419"

if not backbone_weights.exists():
    print("Downloading backbone weights...")
    urlretrieve(backbone_url, backbone_weights)
    print("Saved:", backbone_weights)
else:
    print("Backbone weights already exist:", backbone_weights)

if not chmv2_weights.exists():
    print("Downloading CHMv2 head weights...")
    urlretrieve(chmv2_url, chmv2_weights)
    print("Saved:", chmv2_weights)
else:
    print("CHMv2 head weights already exist:", chmv2_weights)


Backbone weights already exist: /Users/rgcappl037/Documents/New project/electric-pole-analysis/data/weights/chmv2/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth
CHMv2 head weights already exist: /Users/rgcappl037/Documents/New project/electric-pole-analysis/data/weights/chmv2/dinov3_vitl16_chmv2_dpt_head-3703d643.pth


In [7]:
print("Backbone exists:", backbone_weights.exists())
print("CHMv2 head exists:", chmv2_weights.exists())

if backbone_weights.exists():
    print("Backbone size (MB):", round(backbone_weights.stat().st_size / 1024 / 1024, 2))

if chmv2_weights.exists():
    print("CHMv2 head size (MB):", round(chmv2_weights.stat().st_size / 1024 / 1024, 2))


Backbone exists: True
CHMv2 head exists: True
Backbone size (MB): 1156.86
CHMv2 head size (MB): 128.76


In [9]:
from pathlib import Path

project_root = Path("/Users/rgcappl037/Documents/New project/electric-pole-analysis")
input_path = project_root / "data" / "intermediate" / "rgb3_alpha_native.tif"
output_dir = project_root / "data" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_chm_path = output_dir / "chmv2_minimal_rgb3_alpha_native_chm.tif"

weights_dir = project_root / "data" / "weights" / "chmv2"
backbone_weights = weights_dir / "dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
chmv2_weights = weights_dir / "dinov3_vitl16_chmv2_dpt_head-3703d643.pth"

dinov3_repo_dir = Path("/Users/rgcappl037/Documents/New project/dinov3")

print("Input exists:", input_path.exists())
print("DINOv3 repo exists:", dinov3_repo_dir.exists())
print("Backbone weights exist:", backbone_weights.exists())
print("CHMv2 head weights exist:", chmv2_weights.exists())
print("Output path:", output_chm_path)


Input exists: True
DINOv3 repo exists: True
Backbone weights exist: True
CHMv2 head weights exist: True
Output path: /Users/rgcappl037/Documents/New project/electric-pole-analysis/data/outputs/chmv2_minimal_rgb3_alpha_native_chm.tif


In [11]:
import sys
import numpy as np
import matplotlib.pyplot as plt

import rasterio
from rasterio.windows import Window

import torch
import torch.nn.functional as F
from torchvision.transforms import v2
from tqdm.notebook import tqdm


In [13]:
with rasterio.open(input_path) as src:
    print("Band count:", src.count)
    print("Dtypes:", src.dtypes)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Nodata:", src.nodata)
    print("Width x Height:", src.width, src.height)


Band count: 3
Dtypes: ('uint8', 'uint8', 'uint8')
CRS: EPSG:32618
Resolution: (0.5, 0.5)
Nodata: None
Width x Height: 2632 2796


In [15]:
if str(dinov3_repo_dir) not in sys.path:
    sys.path.append(str(dinov3_repo_dir))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

chmv2_model = torch.hub.load(
    str(dinov3_repo_dir),
    "dinov3_vitl16_chmv2",
    source="local",
    weights=str(chmv2_weights),
    backbone_weights=str(backbone_weights),
)
chmv2_model.to(device).eval()

print("CHMv2 model loaded")


Device: cpu


Backbone does not define embed_dims, using [1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024] instead
Backbone does not define input_pad_size, using patch_size=16 instead


CHMv2 model loaded


In [17]:
CHMV2_MEAN = (0.420, 0.411, 0.296)
CHMV2_STD = (0.213, 0.156, 0.143)

image_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=CHMV2_MEAN, std=CHMV2_STD),
    ]
)

print("Transform ready")


Transform ready


In [19]:
if str(dinov3_repo_dir) not in sys.path:
    sys.path.append(str(dinov3_repo_dir))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

chmv2_model = torch.hub.load(
    str(dinov3_repo_dir),
    "dinov3_vitl16_chmv2",
    source="local",
    weights=str(chmv2_weights),
    backbone_weights=str(backbone_weights),
)
chmv2_model.to(device).eval()

print("CHMv2 model loaded")


Device: cpu


Backbone does not define embed_dims, using [1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024] instead
Backbone does not define input_pad_size, using patch_size=16 instead


CHMv2 model loaded


In [21]:
def compute_starts(length: int, tile_size: int, overlap: int) -> list[int]:
    if length <= tile_size:
        return [0]

    step = tile_size - overlap
    starts = list(range(0, length - tile_size + 1, step))
    if starts[-1] != length - tile_size:
        starts.append(length - tile_size)
    return starts


def make_axis_blend(length: int, overlap: int, pin_start: bool, pin_end: bool) -> np.ndarray:
    weights = np.ones(length, dtype=np.float32)
    if overlap <= 0 or length <= 1:
        return weights

    fade = min(overlap, max(1, length // 2))
    edge_floor = 1.0 / (fade + 1)

    if not pin_start:
        weights[:fade] = np.linspace(edge_floor, 1.0, fade, dtype=np.float32)
    if not pin_end:
        weights[-fade:] = np.minimum(weights[-fade:], np.linspace(1.0, edge_floor, fade, dtype=np.float32))
    return weights


def make_blend_mask(tile_height: int, tile_width: int, overlap: int, at_top: bool, at_bottom: bool, at_left: bool, at_right: bool) -> np.ndarray:
    row_weights = make_axis_blend(tile_height, overlap, pin_start=at_top, pin_end=at_bottom)
    col_weights = make_axis_blend(tile_width, overlap, pin_start=at_left, pin_end=at_right)
    return np.outer(row_weights, col_weights).astype(np.float32)


def predict_tile(tile: np.ndarray) -> np.ndarray:
    tile_tensor = torch.from_numpy(tile)
    tile_processed = image_transform(tile_tensor).unsqueeze(0).to(device)

    with torch.inference_mode():
        pred = chmv2_model(tile_processed)
        pred = F.interpolate(pred, size=tile.shape[-2:], mode="bilinear", align_corners=False)

    return pred.squeeze().detach().cpu().to(torch.float32).numpy()


In [ ]:
TILE_SIZE = 512
OVERLAP = 64

with rasterio.open(input_path) as src:
    profile = src.profile.copy()
    height = src.height
    width = src.width
    rgb_preview = src.read([1, 2, 3])
    valid_mask = src.read_masks(1) > 0

    row_starts = compute_starts(height, TILE_SIZE, OVERLAP)
    col_starts = compute_starts(width, TILE_SIZE, OVERLAP)

    prediction_sum = np.zeros((height, width), dtype=np.float32)
    prediction_weight = np.zeros((height, width), dtype=np.float32)

    total_tiles = len(row_starts) * len(col_starts)
    tile_index = 0
    print("Total tiles:", total_tiles)

    for row_idx, row_start in enumerate(tqdm(row_starts, desc="row tiles")):
        for col_idx, col_start in enumerate(col_starts):
            tile_index += 1

            window_height = min(TILE_SIZE, height - row_start)
            window_width = min(TILE_SIZE, width - col_start)
            window = Window(col_off=col_start, row_off=row_start, width=window_width, height=window_height)

            tile = src.read([1, 2, 3], window=window)
            tile_prediction = predict_tile(tile)

            blend_mask = make_blend_mask(
                tile_height=window_height,
                tile_width=window_width,
                overlap=OVERLAP,
                at_top=row_idx == 0,
                at_bottom=row_idx == len(row_starts) - 1,
                at_left=col_idx == 0,
                at_right=col_idx == len(col_starts) - 1,
            )

            row_end = row_start + window_height
            col_end = col_start + window_width

            prediction_sum[row_start:row_end, col_start:col_end] += tile_prediction * blend_mask
            prediction_weight[row_start:row_end, col_start:col_end] += blend_mask

canopy_height = prediction_sum / prediction_weight
canopy_height[~valid_mask] = np.nan

print("Inference complete")


Total tiles: 42


row tiles:   0%|          | 0/7 [00:00<?, ?it/s]